# Lesson 1 | From membrane potential to a minimal computational neuron

Welcome to the first FPGA FlyBrain lesson.

We begin from neurophysiology: membrane potential changes, synaptic input affects a neuron, and under suitable conditions the neuron produces an action potential. Step by step, we will translate those ideas into a **mathematical model → program → digital circuit → neural network running on real hardware**.

Today we will not learn hardware yet. We ask only one question:

> **If we want to study computation in a large neural network, what is the minimum neuronal behavior we need to keep?**

This lesson also follows one rule: **when we introduce a new mechanism, we first make it visible by itself, then combine it with the others.**

## 1. What are you looking at? A Jupyter Notebook

You are reading a **Jupyter Notebook**. It puts textbook material and experiments on the same page:

- **Markdown cells** contain explanations, equations, questions, and conclusions;
- **Code cells** run real Python and let you observe the result immediately.

A Notebook is therefore more than a container for code. It is an **executable textbook**. In this lesson we repeatedly use the pattern: predict → run → inspect the plot → explain. 

## 2. Where are we eventually going? What is an FPGA?

A **Field-Programmable Gate Array (FPGA)** is a chip whose internal digital circuitry can be configured after manufacturing according to our design.

A normal **Central Processing Unit (CPU)** usually executes a sequence of instructions. An FPGA can instead arrange parts of a computation directly as parallel digital circuitry.

Eventually this project will turn neuron state, synaptic events, and network propagation into hardware structures inside an FPGA. For today, put that destination aside and focus only on the neuron model.

## 3. Start from neurophysiology

Keep four familiar observations:

1. there is an electrical potential difference across the neuronal membrane: the **membrane potential**;
2. synaptic input changes membrane potential;
3. without sustained input, membrane potential tends back toward a stable state;
4. under suitable conditions, a neuron can produce an action potential.

A real neuron also contains ion channels, dendrites, transmitter-specific effects, adaptation, synaptic dynamics, cell-type differences, and much more. Lesson 1 does not try to reproduce the whole cell. We keep only a few behaviors that matter for computation.

## 4. What is a model?

A **model** is a purposeful simplification made for the question we are asking. It preserves relationships we currently care about and temporarily ignores other details.

A subway map does not draw every building, but it preserves which stations connect. Likewise, our first neuron model will not reconstruct the full action-potential waveform. It will preserve:

- a membrane-potential state that changes over time;
- the effect of input on that state;
- decay of state over time;
- a spike event when a threshold is reached.

## 5. What is LIF? Read the name literally

We use the classic **Leaky Integrate-and-Fire (LIF)** model.

The three words correspond to three actions:

- **Leaky**: without new input, membrane potential displaced from rest gradually returns toward rest;
- **Integrate**: new input is added to the current membrane state;
- **Fire**: after the membrane state reaches a threshold, the model emits a spike event.

Here a **spike** means only that a firing event occurred at that moment. We are not yet simulating the full rise and fall of a biological action potential or its ion-channel dynamics.

## 6. Write down leak before discussing fire

A passive membrane is commonly written in the form

$$	au_mrac{dV}{dt}=-(V-V_{rest})+R_m I(t)$$

where:

- $V$ is membrane potential;
- $V_{rest}$ is resting membrane potential;
- $	au_m$ is the **membrane time constant**, which controls how long a displacement from rest persists;
- $R_m I(t)$ describes the membrane drive produced by input current.

To build discrete-time intuition first, we use this teaching update rule:

$$V[t+1]=V_{rest}+\alphaig(V[t]-V_{rest}ig)+u[t]$$

Here $u[t]$ is the effective membrane-potential increment produced by input during one time step. In Lesson 1 we absorb factors such as resistance and time-step scaling into $u[t]$, so this is not yet a complete biophysical current equation with physical current units.

When $0<\alpha<1$, only part of the old displacement remains. For pure leak, a useful relation is $\alpha\approx e^{-\Delta t/	au_m}$: an $\alpha$ closer to 1 gives longer memory; a smaller $\alpha$ gives faster decay.

### An easy detail to miss

If we start with $V[0]=V_{rest}$ and set $u[t]=0$, membrane potential remains at rest forever.

The model **still contains leak**, but the plot cannot show it because there is no displacement from rest to decay.

So our first experiment deliberately starts the membrane above rest and then removes all input. That lets us see Leaky by itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def run_leaky_integrator(inputs, *, alpha=0.9, v_rest=-70.0, v0=None):
    """Run only leak + integration. No threshold and no reset yet."""
    inputs = np.asarray(inputs, dtype=float)
    v = v_rest if v0 is None else float(v0)
    voltages = [v]

    for drive in inputs:
        v = v_rest + alpha * (v - v_rest) + drive
        voltages.append(v)

    return np.asarray(voltages)


def plot_input_and_voltage(inputs, voltages, *, title, threshold=None, spike_steps=None):
    inputs = np.asarray(inputs, dtype=float)
    fig, (ax_in, ax_v) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    ax_in.stem(np.arange(len(inputs)), inputs, basefmt=" ")
    ax_in.set_ylabel('input u[t]')
    ax_in.set_title(title)

    ax_v.plot(np.arange(len(voltages)), voltages, marker='o', markersize=3)
    ax_v.axhline(-70.0, linestyle='--', label='V_rest = -70 mV')
    if threshold is not None:
        ax_v.axhline(threshold, linestyle=':', label=f'threshold = {threshold} mV')
    if spike_steps is not None and threshold is not None and len(spike_steps):
        ax_v.scatter(np.asarray(spike_steps) + 1, [threshold] * len(spike_steps), marker='x', s=80, label='spike event')
    ax_v.set_xlabel('time step')
    ax_v.set_ylabel('membrane voltage (mV)')
    ax_v.legend()
    plt.tight_layout()
    plt.show()

## 7. Experiment 1 — No input: see Leak by itself

Start the membrane at `-55 mV`, above the resting value of `-70 mV`, and set every input to zero.

Before running, predict: will the voltage fall linearly, or fall quickly at first and then more slowly?

In [ ]:
leak_inputs = np.zeros(50)
leak_voltage = run_leaky_integrator(
    leak_inputs,
    alpha=0.90,
    v_rest=-70.0,
    v0=-55.0,
)

plot_input_and_voltage(
    leak_inputs,
    leak_voltage,
    title='Experiment 1: leak with no input',
)

### Observe

The upper plot stays at zero input. In the lower plot, the membrane does not remain at `-55 mV`; it gradually returns toward `-70 mV`.

That is **Leaky**. At each step only a fraction of the previous displacement from rest is retained.

Try to answer:

1. Why does the voltage approach `-70 mV` instead of falling without limit?
2. If `alpha` changes from `0.90` to `0.70`, will the return be faster or slower?
3. If the initial value is `-70 mV`, why does the plot become flat?

## 8. Experiment 2 — Periodic pulses: see Integrate and Leak together

Now deliver the same positive input once every 8 time steps.

Each input pushes the membrane upward. Between inputs there is no new drive, so the membrane falls back toward rest. If the next input arrives before the previous effect has fully decayed, it adds on top of the remaining state.

The same plot should now show both actions: **rise on input, decay between inputs.**

In [ ]:
periodic_inputs = np.zeros(70)
periodic_inputs[5::8] = 6.0

periodic_voltage = run_leaky_integrator(
    periodic_inputs,
    alpha=0.85,
    v_rest=-70.0,
    v0=-70.0,
)

plot_input_and_voltage(
    periodic_inputs,
    periodic_voltage,
    title='Experiment 2: periodic inputs reveal integration and leak',
)

### Observe

Do not look only at the peaks. Look closely at the slope between each pair of pulses:

- a pulse arrives: membrane potential jumps upward;
- no input arrives: membrane potential decays toward `V_rest`;
- the next pulse arrives: accumulation continues from whatever state remains.

In LIF, Integrate and Leaky are not two separate phases. They act together in every state update.

## 9. Experiment 3 — Input interval versus membrane time constant

Keep pulse amplitude fixed and change only the interval between pulses.

If inputs arrive quickly, much of the previous input remains. If they arrive slowly, most of the previous effect may have leaked away.

This is a simple experiment in temporal memory.

In [ ]:
def make_periodic_inputs(length, interval, *, amplitude=6.0, start=5):
    x = np.zeros(length)
    x[start::interval] = amplitude
    return x


plt.figure(figsize=(10, 4))
for interval in (4, 8, 16):
    x = make_periodic_inputs(80, interval)
    v = run_leaky_integrator(x, alpha=0.85, v_rest=-70.0)
    plt.plot(np.arange(len(v)), v, label=f'interval = {interval}')

plt.axhline(-70.0, linestyle='--', label='V_rest')
plt.xlabel('time step')
plt.ylabel('membrane voltage (mV)')
plt.title('Experiment 3: same pulse, different arrival interval')
plt.legend()
plt.show()

### Observe

We did not change individual pulse amplitude, and we did not change `alpha`. The only difference is the **timing of the inputs**.

So if the curves accumulate differently, the cause is not a stronger input. It is the neuron's retained state from previous inputs.

This is the central intuition behind the membrane time constant: it controls how long the past can continue to influence the future.

## 10. Experiment 4 — Random input: accumulation and leak alternate

Inputs in a real network are usually not as regular as a metronome. Below we use a fixed random seed to generate a reproducible random input sequence. Both arrival times and pulse amplitudes vary.

The fixed seed ensures that different people running the Notebook get the same sequence, which makes discussion and testing easier.

In [ ]:
rng = np.random.default_rng(7)
random_events = rng.random(100) < 0.12
random_amplitudes = rng.uniform(4.0, 7.0, size=100)
random_inputs = random_events * random_amplitudes

random_voltage = run_leaky_integrator(
    random_inputs,
    alpha=0.88,
    v_rest=-70.0,
)

plot_input_and_voltage(
    random_inputs,
    random_voltage,
    title='Experiment 4: random input events',
)

### Observe

Where input events cluster, membrane potential is more likely to accumulate. Where events are sparse, the membrane has more time to decay toward rest.

We can now describe the temporal behavior of LIF in one complete statement:

> Current membrane potential depends not only on whether an input arrives now, but also on how much state past inputs left behind and how much of that state leaked away while waiting.

## 11. Experiment 5 — Only now add threshold and reset

So far we have built a **Leaky Integrator**: it leaks and integrates, but it does not fire.

Now we add the final word in LIF: **Fire**. The rule is:

1. compute `candidate_v` with the same leak + integration update;
2. if `candidate_v >= threshold`, record one spike;
3. after the spike, set the state actually stored for the next step to `reset`.

We must distinguish two values:

- `candidate_v`: the next-state candidate before threshold/reset;
- `stored_v`: the state actually retained after the threshold/reset rule.

In [ ]:
def run_lif(inputs, *, alpha=0.9, v_rest=-70.0, v0=None, threshold=-50.0, reset=-70.0):
    inputs = np.asarray(inputs, dtype=float)
    v = v_rest if v0 is None else float(v0)
    stored_voltage = [v]
    candidate_voltage = []
    spike_steps = []

    for t, drive in enumerate(inputs):
        candidate_v = v_rest + alpha * (v - v_rest) + drive
        candidate_voltage.append(candidate_v)

        if candidate_v >= threshold:
            spike_steps.append(t)
            v = reset
        else:
            v = candidate_v

        stored_voltage.append(v)

    return {
        'inputs': inputs,
        'candidate_voltage': np.asarray(candidate_voltage),
        'stored_voltage': np.asarray(stored_voltage),
        'spike_steps': np.asarray(spike_steps, dtype=int),
    }


fire_inputs = np.zeros(80)
fire_inputs[3::4] = 7.0
trace = run_lif(fire_inputs, alpha=0.90, threshold=-50.0, reset=-70.0)

plot_input_and_voltage(
    trace['inputs'],
    trace['stored_voltage'],
    title='Experiment 5: threshold, spike, and reset',
    threshold=-50.0,
    spike_steps=trace['spike_steps'],
)

print('spike steps:', trace['spike_steps'])

## 12. Read state step by step around the first spike

Plots are good for intuition, while numbers help us verify update order. Print only a few time steps around the first spike.

In [ ]:
first_spike = int(trace['spike_steps'][0])
for t in range(first_spike - 2, first_spike + 2):
    did_spike = t in set(trace['spike_steps'].tolist())
    print({
        't': t,
        'input': trace['inputs'][t],
        'candidate_v': trace['candidate_voltage'][t],
        'stored_v': trace['stored_voltage'][t + 1],
        'spike': did_spike,
    })

This introduces an idea that will later map directly to digital hardware: the computed **next state** and the state ultimately written into a state register do not have to be identical.

At a spike step, `candidate_v` reaches threshold, while `stored_v` has already been returned to `-70 mV` by the reset rule. Later lessons will turn this update order into an explicit, testable specification.

## 13. Try It — Change only one factor at a time

Write your prediction before running. Pick one:

- Experiment 1: change `alpha=0.90` to `0.70` and predict the decay curve;
- Experiment 2: keep pulse amplitude fixed and change the interval from 8 to 4;
- Experiment 3: keep the input interval fixed and change `alpha` from `0.85` to `0.95`;
- Experiment 4: change random-event probability from `0.12` to `0.25`;
- Experiment 5: change threshold from `-50 mV` to `-45 mV`.

Change one factor at a time so that you can identify what caused the behavioral change.

## 14. AI Task

You may ask AI to:

- generate additional input patterns, such as clustered burst input;
- compare several `alpha` values or input intervals;
- turn the experiments into automated tests;
- check whether the implementation matches the equations.

Give AI one constraint:

> Do not silently change the current update order, threshold rule, or reset rule. If a change seems useful, state clearly that it is a specification change rather than ordinary code refactoring.

AI can help write the implementation, but the meaning of the model must remain an explicit human decision.

## 15. Human Check

Without AI, you should be able to explain:

- why there is no visible leak curve when `V = V_rest` and there is no input;
- why starting above `V_rest` reveals decay back toward rest;
- why membrane potential falls between periodic pulses but can still accumulate across pulses;
- why input interval and membrane time constant jointly control temporal integration;
- which regions of a random-input trace are dominated by integration and which by leak;
- why `candidate_v` and `stored_v` may differ;
- what Leaky / Integrate / Fire each means.

## 16. Engineering Handoff

This Notebook is still a teaching prototype. After model semantics are confirmed, the formal reference implementation should live in:

`python/reference/lif_float.py`

The Notebook should later import the formal module instead of permanently maintaining a second implementation with slightly different behavior.

Before RTL implementation, at least these semantics must be frozen: update order, threshold comparison, reset rule, numerical representation, and time-step definition.

## 17. Project Trace

- Lesson ID: `LSN-001`
- Engineering slice: `RMD-001`
- Product requirement/design: `FR1 / DP1`
- Initial tests: `T-001 ~ T-004`

These IDs keep lessons, requirements, code, and tests traceable over time.

## 18. Exit Ticket

Before continuing, you should be able to:

1. explain in your own words why a scientific model is a purposeful simplification;
2. expand and explain **Leaky Integrate-and-Fire (LIF)**;
3. point to leak in the zero-input experiment;
4. point to both integration and leak in the periodic-input experiment;
5. explain why shorter input intervals make it easier to retain and accumulate past input;
6. explain why membrane potential rises and falls under random input;
7. manually step through several updates and explain when threshold/reset occurs.

If those ideas feel natural, the next lesson asks:

> When `-70 mV`, `alpha`, and input increments enter real digital hardware, how are those numbers represented and stored?